In [1]:
# Importação de bibliotecas
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
# Leitura dos dados da dataframe
dados = pd.read_csv('../dados/dados_pos_an_exp.csv')

In [3]:
# Tratamentos e edições da dataframe
# Transformar MS em MIN
dados['duration_ms'] = dados['duration_ms'] / 60000
# Renomear a coluna duration_ms para duration_min
dados.rename(columns={'duration_ms': 'duration_min'}, inplace=True)
# Transformar os gêneros em listas
dados['genres'] = dados['genres'].apply(eval)
# Criar a coluna de Quantidade de Gêneros
dados['qt_genres'] = dados['genres'].apply(len)

In [4]:
# Criação dos dummies da dataframe
mlb = MultiLabelBinarizer()
genres_dummies = pd.DataFrame(mlb.fit_transform(dados['genres']), columns=mlb.classes_)

In [5]:
# Dropar colunas irrelevantes
dados = dados.drop(columns=['music', 'genres', 'artist'])

In [6]:
# Divisão para futuro escalonamento
colunas_nao_numericas = list(dados.select_dtypes(exclude=['int64', 'float64']).columns)
irrelevantes = colunas_nao_numericas + ['duration_min', 'qt_genres']
alvo = dados['liked']
numericas = dados.drop(columns=irrelevantes + ['liked'])

In [8]:
# Escalonamento
scaler = StandardScaler()
numericas_scaled = scaler.fit_transform(numericas)
joblib.dump(scaler, '../dados/scaler.pkl')
numericas_dataframe = pd.DataFrame(numericas_scaled, columns=numericas.columns)

In [9]:
# Concatenação
dados_scaled = pd.concat([dados[irrelevantes].reset_index(drop=True), numericas_dataframe.reset_index(drop=True), alvo.reset_index(drop=True)], axis=1)

In [10]:
# Concatenação final
dados_final = pd.concat([dados_scaled.drop(columns=['liked']), genres_dummies, dados['liked']], axis=1)
dados_final

,duration_min,qt_genres,music_popularity,artist_popularity,followers,acid jazz,acid rock,acoustic pop,alternative metal,alternative rock,...,soul blues,southern gothic,southern rock,stoner rock,surf rock,symphonic metal,synthpop,traditional country,yacht rock,liked
0,2.965767,1,0.722166,0.487563,-0.076675,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,4.097100,1,0.781118,0.387802,-0.317030,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,3.811767,2,1.046404,0.587324,0.012511,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4.255550,1,1.046404,-0.210762,-0.416673,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,3.765100,1,0.751642,-0.011241,-0.281761,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,4.054433,3,0.162119,-0.809326,-0.476202,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
138,4.197333,3,0.309500,-0.809326,-0.476202,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
139,5.427100,3,0.250547,-0.809326,-0.476202,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
140,3.663100,3,-0.781118,-0.809326,-0.476202,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
# Exportação dos dados tratados
dados_final.to_pickle('../dados/dados_tratados.pkl')